In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install qdrant-client ijson --quiet

In [ ]:
import ijson

file_path = "/kaggle/input/datasets/ahmedezzattaha/mechrabot-qdrant-ready-data/mechrabot_qdrant_ready.json"

with open(file_path, "r", encoding="utf-8") as file:
    for item in ijson.items(file, "item"): 
        print("✅ تم سحب أول Chunk بنجاح!")
        print("-" * 30)
        print("أسماء الـ Keys الموجودة عندك:")
        print(list(item.keys()))
        print("-" * 30)
        break  # بنوقف اللوب بعد أول عنصر فوراً

In [ ]:
import ijson
from qdrant_client import QdrantClient, models
from tqdm import tqdm

# 1. الاتصال
client = QdrantClient(
    url="YOUR_QDRANT_URL", 
    api_key="YOUR_QDRANT_API_KEY"
)
col_name = "mechrabot_Vdb_1"

# 2. إنشاء وتجهيز الـ Collection (الخطوة اللي كانت ناقصة)
client.recreate_collection(
    collection_name=col_name,
    vectors_config={
        "dense": models.VectorParams(size=1024, distance=models.Distance.COSINE),
        "colbert": models.VectorParams(
            size=1024, 
            distance=models.Distance.COSINE, 
            multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM)
        ),
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams()}
)
print("✅ تم إنشاء غرفة الداتا بيز وتجهيزها بنجاح!")

# 3. الدالة المُولدة
def get_points():
    with open("/kaggle/input/datasets/ahmedezzattaha/mechrabot-qdrant-ready-data/mechrabot_qdrant_ready.json", "r") as f:
        for c in tqdm(ijson.items(f, 'item'), total=2790, desc="🚀 جاري سحب الـ Chunks"):
            yield models.PointStruct(
                id=c["chunk_id"],
                vector={
                    "dense": [float(x) for x in c["dense_vec"]],
                    "colbert": [[float(x) for x in t] for t in c["colbert_vecs"]],
                    "sparse": models.SparseVector(indices=[int(k) for k in c["sparse_vec"]], values=[float(v) for v in c["sparse_vec"].values()])
                },
                payload={"content": c["content"], **c["meta"]}
            )

# 4. الرفع مع مراقبة التقدم
print("⏳ بدأنا المعركة.. تابع شريط التحميل تحت:")
client.upload_points(
    collection_name=col_name, 
    points=get_points(), 
    batch_size=2, 
    parallel=2
)
print("\n✅ مبروك يا هندسة! الداتا كلها وصلت بسلام.")